In [ ]:
# =============================================================================
# POLICY GRADIENTS — learn the policy directly, not a Q-table
# =============================================================================
#
# So far (notebook 23):
#   Q-learning / SARSA / DQN learn VALUES: "how good is this state–action?"
#   Then the policy is derived: π(s) = argmax_a Q(s, a)
#
# Policy gradients flip that:
#   Learn the POLICY itself: π_θ(a | s) = "probability of action a in state s"
#   θ = neural-net (or other) parameters. We nudge θ so good actions become
#   more likely and bad actions less likely.
#
# Why bother?
#   1) Continuous actions (steering angle, robot torque) — argmax over infinite
#      actions is hard; a policy that OUTPUTS the action is natural.
#   2) Stochastic policies — sometimes you NEED randomness (rock-paper-scissors,
#      exploration). Softmax over Q is one way; a learned π is another.
#   3) End-to-end: one network from state → action distribution.
#
# Mental model:
#   Value methods  = grade every move, then pick the best grade.
#   Policy methods = practice the habit directly ("when I see X, do Y more").
#
#
# ---------------------------------------------------------------------------
# 1) THE POLICY GRADIENT THEOREM — the math that makes this legal
# ---------------------------------------------------------------------------
#
# Goal: maximize expected return J(θ) = E_τ~π_θ [ G(τ) ]
#   τ (trajectory) = s0, a0, r1, s1, a1, r2, ... until episode ends
#   G(τ)           = total discounted reward of that whole episode
#
# We want ∇_θ J(θ) — which way to push θ to raise average return.
#
# Naive problem: the environment is a black box. You cannot differentiate
# through "the world responded with this next state." So you cannot just
# backprop through the simulator like a normal supervised loss.
#
# THE THEOREM (intuitive form):
#
#   ∇_θ J(θ)  ∝  E_τ [  Σ_t  ∇_θ log π_θ(a_t | s_t)  ·  G_t  ]
#
# English, word by word:
#   "Average, over trajectories you actually play,
#      for every step t in the episode:
#         take the gradient of log-probability of the action you took,
#         and SCALE it by how good the future turned out (G_t)."
#
# Why log π?
#   Score-function / log-derivative trick:
#     ∇_θ π = π · ∇_θ log π
#   So the unknown π in the expectation cancels into "sample from π, then
#   weight by ∇ log π." You only need samples — no world derivative.
#
# Intuition with a joke:
#   You tell a joke. Crowd laughs hard (high G). → Increase probability of
#   that joke style in that situation.
#   Crowd is silent (low / negative G). → Decrease that probability.
#
# Important: G_t is the RETURN FROM TIME t onward, not just the immediate
# reward. Delayed credit: the punchline later still shapes earlier setup lines.
#
# This is the foundation. Every algorithm below is "how do we estimate this
# expectation with less noise / less waiting?"
#
#
# ---------------------------------------------------------------------------
# 2) REINFORCE — Monte Carlo policy gradient (the simplest version)
# ---------------------------------------------------------------------------
#
# REINFORCE = "REward Increment = Nonnegative Factor × Offset Reinforcement"
# (Williams, 1992). Fancy name, simple algorithm.
#
# Recipe:
#   1. Play ONE full episode with current π_θ  → get (s0,a0,r1), (s1,a1,r2), ...
#   2. For each time t, compute return G_t = r_{t+1} + γ r_{t+2} + γ² r_{t+3} + ...
#   3. Update:
#        θ ← θ + α · Σ_t  ∇_θ log π_θ(a_t | s_t) · G_t
#   4. Repeat.
#
# "Monte Carlo" means: we wait until the episode ENDS, then use the real
# total return. No bootstrapping from a value network. Pure experience.
#
# Tiny example — CartPole (balance a stick):
#   Episode scored G = +200 (stayed up long). Left at t=3 was part of that.
#   → bump up log π(left | that state) a bit.
#   Episode scored G = +12 (fell fast). Right at t=1 was part of that.
#   → push log π(right | that state) down.
#
# Pros:
#   - Simple, unbiased (in expectation, points the right way).
#   - Easy to implement.
#
# Cons (why people invent Actor-Critic):
#   - HIGH VARIANCE: one lucky episode with huge G can yank θ hard even if
#     most actions were mediocre. Training is noisy / sample-hungry.
#   - Must wait for episode end (slow credit, bad for very long tasks).
#   - No baseline → every positive return pushes "up," even average moves.
#
# Optional upgrade: BASELINE
#   Subtract a state-dependent baseline b(s) that does not depend on the
#   action. Common choice: b(s) ≈ V(s) (how good is this state on average?).
#
#   θ ← θ + α · Σ_t  ∇_θ log π(a_t|s_t) · (G_t − b(s_t))
#
#   Why still valid? E[∇ log π · b(s)] = 0 under mild conditions — baseline
#   doesn't bias the gradient, only shrinks variance.
#   Intuition: reward relative to "what I usually get here," not absolute.
#
#
# ---------------------------------------------------------------------------
# 3) ACTOR–CRITIC — two brains: one acts, one grades
# ---------------------------------------------------------------------------
#
# Split the job:
#
#   ACTOR  = the policy π_θ(a | s)
#            "What should I do?"
#            Updated with policy-gradient-style steps.
#
#   CRITIC = a value function V_w(s)  (or Q_w(s,a))
#            "Was that good?"
#            Updated like TD learning (bootstrapped value targets).
#
# Flow of one step:
#   1. Actor picks a_t ~ π_θ(· | s_t)
#   2. Environment returns r_{t+1}, s_{t+1}
#   3. Critic judges: "was this step better or worse than I expected?"
#        δ_t = r_{t+1} + γ V_w(s_{t+1}) − V_w(s_t)     ← TD error
#   4. Critic learns: nudge V_w toward making δ smaller
#        (usual TD / MSE on the bootstrap target).
#   5. Actor learns: nudge π so that actions with positive δ become more likely
#        θ ← θ + α · ∇_θ log π_θ(a_t | s_t) · δ_t
#
# Why better than plain REINFORCE?
#   - Learn ONLINE every step — no waiting for episode end.
#   - Critic's V acts as a baseline → lower variance than raw G_t.
#   - Still uses the policy-gradient idea (∇ log π), just with a smarter
#     "how good was this?" signal.
#
# Analogy:
#   Actor  = the player practicing moves.
#   Critic = the coach shouting "that was better than usual!" or "worse!"
#   The player does not wait until the final scoreboard; coach gives feedback
#   after each play.
#
# Caveat: critic can be wrong early on → biased targets. In practice you
#   train both together and it usually works.
#
#
# ---------------------------------------------------------------------------
# 4) A2C — Advantage Actor–Critic
# ---------------------------------------------------------------------------
#
# Advantage = "how much BETTER was this action than the average action here?"
#
#   A(s, a) = Q(s, a) − V(s)
#
# English:
#   Q(s,a) = expected return if I take a now, then follow the policy
#   V(s)   = expected return if I just follow the policy from s (average action)
#   A(s,a) = extra credit (or blame) for choosing a specifically
#
#   A > 0  → this action beat the average → do it more
#   A < 0  → this action was worse than average → do it less
#   A ≈ 0  → typical → little change
#
# Why advantage helps:
#   Absolute returns are noisy and state-dependent (some states always score
#   high). Advantage centers the signal: only relative quality of the action.
#
# In practice we rarely store both Q and V. Common estimate from one step:
#
#   Â_t ≈ δ_t = r_{t+1} + γ V(s_{t+1}) − V(s_t)
#
# That TD error IS an estimate of the advantage. So one-step Actor–Critic
# with δ is already "advantage actor–critic" in spirit.
#
# A2C (synchronous Advantage Actor–Critic) — the practical package:
#   - Use advantage Â (often n-step returns: look ahead n steps, then bootstrap)
#   - Actor loss ≈ − log π(a|s) · Â     (ascent on expected advantage)
#   - Critic loss ≈ (return_target − V(s))²
#   - Often add entropy bonus: encourage π to stay a bit random (explore)
#   - "Synchronous": several workers collect experience, then one update
#     (simpler cousin of A3C's async updates)
#
# Compared to REINFORCE:
#   REINFORCE: wait for full G, high variance, one policy net
#   A2C:       critic grades every few steps, lower variance, actor + critic
#
# Compared to DQN (value-based):
#   DQN:  learn Q, act with ε-greedy / argmax
#   A2C:  learn π and V together; action comes from sampling π
#
#
# ---------------------------------------------------------------------------
# 5) POLICY PARAMETERIZATIONS — how π_θ actually looks
# ---------------------------------------------------------------------------
#
# The network must OUTPUT a distribution over actions (or parameters of one).
# Choice depends on the action space.
#
# (A) DISCRETE actions  (left / right / jump — CartPole, Atari buttons)
#     Network → logits (one number per action)
#     π(a | s) = softmax(logits)
#     Sample a ~ Categorical(π)
#     ∇ log π is standard cross-entropy style gradient on the taken action.
#
# (B) CONTINUOUS actions  (steer ∈ [-1,1], joint torque)
#     Common: Diagonal Gaussian
#       Network outputs mean μ_θ(s)  (and often log-std)
#       a ~ Normal(μ, σ)
#       log π(a|s) = Gaussian log-density
#     Sometimes squash with tanh for bounded actions (as in SAC / some PPO).
#
# (C) DETERMINISTIC policy  (a = μ_θ(s), no randomness in the policy itself)
#     Used in DDPG / TD3. Exploration comes from adding noise outside.
#     Gradients use a different trick (deterministic policy gradient) —
#     sibling idea, not classical REINFORCE.
#
# (D) Extra knobs people add:
#     - Temperature / entropy: softer π = more explore
#     - Shared trunk: actor and critic share early layers, separate heads
#     - Softmax vs sparsemax / other discrete distributions (rare in intro)
#
# Rule of thumb:
#   Buttons / menus     → Categorical (softmax)
#   Real-valued controls → Gaussian (or Beta for [0,1] bounds)
#
#
# ---------------------------------------------------------------------------
# BIG PICTURE (how these pieces nest)
# ---------------------------------------------------------------------------
#
#   Policy Gradient Theorem  →  "∇ J ≈ E[ ∇ log π · (something good) ]"
#                |
#                +-- REINFORCE: something = full Monte Carlo return G_t
#                |
#                +-- Actor–Critic: something = TD error / critic signal
#                        |
#                        +-- A2C: something = Advantage Â = Q − V
#                                            (estimated, often via δ or n-step)
#
# Parameterization is orthogonal: any of the above can use softmax or Gaussian.
#
# Next time you see code: look for
#   (1) a policy net that outputs probs / μ,σ
#   (2) log_prob of the chosen action
#   (3) a scalar weight (G, δ, or Â)
#   (4) loss ≈ − log_prob * weight   (+ critic MSE)
# That pattern IS policy gradients.


In [1]:
# Setup
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical, Normal
import gym
import matplotlib.pyplot as plt
from collections import deque

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'gym'

In [ ]:
# Policy Network (Discrete)
class PolicyNet(nn.Module):
    def __init__(self, state_dim, n_actions, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, n_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        logits = self.fc3(x)
        return logits

    def act(self, state):
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        logits = self.forward(state)
        dist = Categorical(logits=logits)
        action = dist.sample()
        return action.item(), dist.log_prob(action).squeeze(0)